# Data Preprocessing and Feature Engineering

## Objective
The objective of this notebook is to clean and preprocess the book data stored
in the SQLite database and prepare it for machine learning modeling.

This includes:
- Handling missing values
- Removing duplicates
- Feature engineering
- Converting categorical variables into numerical format
- Saving the final processed dataset


## Import Required Libraries

In [21]:
import pandas as pd
import sqlite3
from sklearn.preprocessing import LabelEncoder, StandardScaler


## Load Data from SQLite Database
The book data is loaded directly from the SQLite database created earlier.

In [22]:
conn = sqlite3.connect("data/books_database.db")
df = pd.read_sql("SELECT * FROM books", conn)
conn.close()

df.head()

,title,price,availability,rating
0,A Light in the Attic,Â51.77,In stock,Three
1,Tipping the Velvet,Â53.74,In stock,One
2,Soumission,Â50.10,In stock,One
3,Sharp Objects,Â47.82,In stock,Four
4,Sapiens: A Brief History of Humankind,Â54.23,In stock,Five


## Data Overview
We inspect the dataset to understand its structure and identify potential issues.

In [23]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         1000 non-null   object
 1   price         1000 non-null   object
 2   availability  1000 non-null   object
 3   rating        1000 non-null   object
dtypes: object(4)
memory usage: 31.4+ KB


,title,price,availability,rating
count,1000,1000,1000,1000
unique,999,903,1,5
top,The Star-Touched Queen,Â44.18,In stock,One
freq,2,3,1000,226


## Handling Missing Values
We check for missing values and handle them appropriately.

In [24]:
df.isnull().sum()

title           0
price           0
availability    0
rating          0
dtype: int64

In [25]:
# No missing values in this dataset, but this step is included for completeness
df = df.dropna()

## Remove Duplicate Records
Duplicate records can negatively affect model performance.

In [26]:
df.duplicated().sum()
df = df.drop_duplicates()


## Feature Engineering – Convert Ratings to Numerical Values
Book ratings are stored as text and need to be converted into numeric values.

In [27]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating_numeric"] = df["rating"].map(rating_map)


## Feature Engineering – Availability Encoding
Availability information is converted into binary format.

In [28]:
df["availability_binary"] = df["availability"].apply(
    lambda x: 1 if "In stock" in x else 0
)

## Drop Irrelevant Columns
Text-based columns not useful for modeling are removed.

In [29]:
df_model = df.drop(columns=["title", "availability", "rating"])
df_model.head()

,price,rating_numeric,availability_binary
0,Â51.77,3,1
1,Â53.74,1,1
2,Â50.10,1,1
3,Â47.82,4,1
4,Â54.23,5,1


In [30]:
df_model['price'] = df_model['price'].replace(r'[^\d.]', '', regex=True)
df_model['price'] = pd.to_numeric(df_model['price'])
scaler = StandardScaler()
df_model[["price"]] = scaler.fit_transform(df_model[["price"]])

## Feature Scaling
Numerical features are scaled to improve machine learning model performance.


In [31]:
scaler = StandardScaler()
df_model[["price"]] = scaler.fit_transform(df_model[["price"]])

## Final Dataset Review
The final dataset is now clean and ready for machine learning.


In [32]:
df_model.head()
df_model.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   price                1000 non-null   float64
 1   rating_numeric       1000 non-null   int64  
 2   availability_binary  1000 non-null   int64  
dtypes: float64(1), int64(2)
memory usage: 23.6 KB


## Save Processed Dataset
The processed dataset is saved as a CSV file for machine learning modeling.


In [33]:
df_model.to_csv("data/books_processed.csv", index=False)

## Summary
In this notebook, the book dataset was successfully cleaned and preprocessed.
Key steps included removing duplicates, converting categorical variables into
numerical format, feature scaling, and saving the final dataset.

This processed data will be used in the next stage to build and evaluate
machine learning models.
